# 🧠 EXACT 2026 - Track 1: Logic-Based Educational QA
### Hướng dẫn chạy thử nghiệm Pipeline trên Google Colab với Google Drive Cache

Notebook này hướng dẫn bạn thiết lập môi trường và chạy thử nghiệm hệ thống Neuro-Symbolic QA (Track 1) trên Google Colab sử dụng GPU. Để tối ưu hóa thời gian, chúng ta sẽ **lấy trực tiếp file `.whl` của `llama-cpp-python` và các file mô hình `.gguf` từ thư mục `Colab_Cache` trên Google Drive** thay vì phải tải và biên dịch lại từ đầu.

---

## 1. Kết nối Google Drive và kiểm tra GPU

Chạy cell dưới đây để kết nối với Google Drive của bạn (nhằm truy cập thư mục `Colab_Cache`). Đồng thời kiểm tra xem Colab đã nhận GPU chưa.

*(Lưu ý: Đi tới **Runtime** -> **Change runtime type** -> chọn **T4 GPU** trước khi chạy)*

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra GPU
!nvidia-smi

## 2. Clone Repository và chuyển sang nhánh `test/track1`

In [ ]:
# Clone repo từ Github
!git clone https://github.com/AIVIETNAM-AIO-Triet-Descartes/EXACT2026-NeuroSymbolic-QA.git

# Chuyển con trỏ dòng lệnh vào thư mục dự án (sử dụng %cd để thay đổi thư mục vĩnh viễn cho kernel)
%cd /content/EXACT2026-NeuroSymbolic-QA

# Chuyển sang nhánh test/track1
!git checkout test/track1

## 3. Cài đặt các thư viện phụ thuộc (Dependencies)

Chúng ta sẽ cài đặt các thư viện trong `requirements.txt`. Riêng đối với `llama-cpp-python`, ta sẽ cài đặt trực tiếp từ file `.whl` đã được biên dịch sẵn trong thư mục `Colab_Cache` trên Google Drive để tránh tốn thời gian compile.

In [ ]:
# Đảm bảo đứng đúng thư mục dự án và cài đặt dependencies
%cd /content/EXACT2026-NeuroSymbolic-QA

!pip install -r requirements.txt

# Cài đặt llama-cpp-python từ file wheel (.whl) lưu trên Google Drive để tiết kiệm thời gian
!pip install /content/drive/MyDrive/Colab_Cache/llama_cpp_python-0.3.23-py3-none-linux_x86_64.whl

## 4. Chuẩn bị Mô hình Qwen 2.5 7B GGUF

Bạn có hai sự lựa chọn để trỏ đến mô hình GGUF:

### **Cách 1 (Khuyên Dùng): Copy mô hình vào bộ nhớ đệm Colab (Chạy nhanh hơn)**
Sao chép mô hình từ Google Drive sang ổ cứng của Colab (mất khoảng 1 - 2 phút). Tốc độ đọc file lúc suy luận (inference) sẽ nhanh hơn đáng kể.

In [ ]:
# Di chuyển vào thư mục dự án
%cd /content/EXACT2026-NeuroSymbolic-QA

# Sao chép các file GGUF từ Drive vào thư mục hiện tại (Lưu ý: Đợi lệnh chạy xong 100%)
!cp /content/drive/MyDrive/Colab_Cache/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf .
!cp /content/drive/MyDrive/Colab_Cache/qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf .

# Kiểm tra file đã tồn tại ở local hay chưa
import os
if os.path.exists("./qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf"):
    print("✅ Copy model sang bộ nhớ Colab thành công!")
else:
    print("❌ Copy model thất bại hoặc chưa hoàn thành. Hãy chạy lại hoặc dùng Cách 2.")

### **Cách 2: Đọc trực tiếp từ Google Drive (Không cần copy)**
Nếu bạn không muốn đợi copy, bạn có thể chạy trực tiếp bằng cách trỏ đường dẫn mô hình vào Drive. 

*(Lưu ý: Thời gian load model lần đầu sẽ lâu hơn khoảng 1-2 phút do giới hạn băng thông Google Drive)*

In [ ]:
# Kiểm tra xem model có tồn tại trên Drive của bạn hay không
import os
drive_model_path = "/content/drive/MyDrive/Colab_Cache/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf"
if os.path.exists(drive_model_path):
    print("✅ Tìm thấy model trên Google Drive!")
else:
    print("❌ Không tìm thấy model tại:", drive_model_path)

## 5. Chạy thử nghiệm Pipeline (Track 1)

Sử dụng script `scripts/run_track1.py` để chạy pipeline.

### 5.1 Chạy thử nhanh với 5 mẫu đầu tiên (Sử dụng Cách 1 - Model Local)

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_test.json \
    --model ./qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf \
    --max-samples 5 \
    --gpu-layers -1 \
    --evaluate

### 5.2 Chạy thử nhanh với 5 mẫu đầu tiên (Sử dụng Cách 2 - Đọc trực tiếp từ Google Drive)
Dùng cách này nếu Cách 1 báo lỗi thiếu model file.

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_test.json \
    --model /content/drive/MyDrive/Colab_Cache/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf \
    --max-samples 5 \
    --gpu-layers -1 \
    --evaluate

### 5.3 Chạy đánh giá trên dải dữ liệu cụ thể (ví dụ: Mẫu 50 đến 100)

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_50_100.json \
    --model ./qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf \
    --start-sample 50 \
    --end-sample 100 \
    --gpu-layers -1 \
    --evaluate

### 5.4 Chạy toàn bộ Dataset (411 mẫu / ~808 câu hỏi)

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_full.json \
    --model ./qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf \
    --gpu-layers -1 \
    --evaluate

## 6. Xem Kết Quả Đầu Ra
Sau khi chạy xong, các kết quả dự đoán và đánh giá chi tiết (bao gồm cả độ chính xác Accuracy, số câu trả lời đúng/sai của mô hình) sẽ được lưu tại thư mục `output/`.

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

# Hiển thị 30 dòng đầu của file dự đoán để kiểm tra cấu trúc đầu ra
!head -n 30 output/predictions_test.json